# F1 RAG — Data Loading Practice

This notebook walks through each of the three data loaders:
1. `load_track_data.py` — circuit geometry (works offline, no F1 API needed)
2. `load_qualifying.py` — FastF1 session data (requires internet)
3. `load_openf1.py` — OpenF1 weather + race control (requires internet)

**Run the track data section first** — it works anywhere and gives you a feel for the data shape before pulling live session data.

## Setup

In [1]:
import sys
import os

# Add the project root and loader directory to path so imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../RAG_data_layers/F1 Project Thoughts'))

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Setup complete.')

Setup complete.


---
## Part 1: Track Data

No F1 API needed — pulls from GitHub (bacinger GeoJSON + TUMFTM CSV).  
This is the circuit geometry layer of our RAG system.

In [2]:
from load_track_data import (
    load_circuit_metadata,
    load_circuit_geometry,
    load_all_track_data,
    describe_circuit,
)

print('Track data functions imported.')

Track data functions imported.


### 1a. Load circuit metadata (bacinger)

In [3]:
# Load metadata for Monza
meta = load_circuit_metadata('Monza')

# What keys do we get?
print('Keys:', list(meta.keys()))
print()

# Print each field
for k, v in meta.items():
    if k == 'gps_coordinates':
        print(f'  gps_coordinates: {len(v)} points, first = {v[0]}')
    else:
        print(f'  {k}: {v}')

Keys: ['name', 'location', 'circuit_id', 'length_m', 'altitude_m', 'first_gp_year', 'opened', 'gps_coordinates', 'coordinate_count']

  name: Autodromo Nazionale Monza
  location: Monza
  circuit_id: it-1922
  length_m: 5793
  altitude_m: 142
  first_gp_year: 1950
  opened: 1922
  gps_coordinates: 125 points, first = [9.281223, 45.618975]
  coordinate_count: 125


In [4]:
# Try a few different circuits and compare their metadata
circuits = ['Monza', 'Monaco', 'Spa', 'Silverstone', 'Jeddah']

rows = []
for c in circuits:
    m = load_circuit_metadata(c)
    rows.append({
        'Circuit':      m['name'],
        'Length (m)':   m['length_m'],
        'Altitude (m)': m['altitude_m'],
        'First GP':     m['first_gp_year'],
        'GPS Points':   m['coordinate_count'],
    })

pd.DataFrame(rows).set_index('Circuit')

,Length (m),Altitude (m),First GP,GPS Points
Circuit,,,,
Autodromo Nazionale Monza,5793,142,1950,125
Circuit de Monaco,3337,47,1929,160
Circuit de Spa-Francorchamps,7004,413,1950,153
Silverstone Circuit,5891,196,1950,135
Jeddah Corniche Circuit,6175,12,2021,152


### 1b. Load circuit geometry (TUMFTM)

Higher resolution x/y coordinates in meters + track width.  
Available for: Monza, Silverstone, Spa, Suzuka, Barcelona, Zandvoort, Austin, Melbourne.

In [5]:
# Load Monza geometry
geom = load_circuit_geometry('Monza')

print('Type:', type(geom))
print('Shape:', geom.shape)
print('Columns:', list(geom.columns))
print()
geom.head(10)

Type: <class 'pandas.core.frame.DataFrame'>
Shape: (1159, 4)
Columns: ['x_m', 'y_m', 'w_tr_right_m', 'w_tr_left_m']



,x_m,y_m,w_tr_right_m,w_tr_left_m
0,-0.320,1.088,5.739,5.932
1,0.168,6.062,5.735,5.929
2,0.656,11.037,5.731,5.926
3,1.144,16.011,5.727,5.923
4,1.631,20.985,5.723,5.920
5,2.117,25.960,5.719,5.917
6,2.603,30.934,5.715,5.914
7,3.089,35.909,5.711,5.911
8,3.575,40.883,5.707,5.908
9,4.061,45.857,5.703,5.905


In [6]:
# Basic stats on track geometry
geom['total_width_m'] = geom['w_tr_right_m'] + geom['w_tr_left_m']

print('Track width stats (meters):')
print(geom['total_width_m'].describe())
print()
print(f'Narrowest point: {geom["total_width_m"].min():.1f}m')
print(f'Widest point:    {geom["total_width_m"].max():.1f}m')
print(f'Average width:   {geom["total_width_m"].mean():.1f}m')

Track width stats (meters):
count   1159.000
mean       9.368
std        1.030
min        7.516
25%        8.697
50%        9.004
75%        9.977
max       12.421
Name: total_width_m, dtype: float64

Narrowest point: 7.5m
Widest point:    12.4m
Average width:   9.4m


In [7]:
# What happens with a circuit NOT in TUMFTM?
geom_monaco = load_circuit_geometry('Monaco')
print('Monaco geometry:', geom_monaco)  # Should be None with a helpful note

  Note: Monaco not in TUMFTM database. Available: ['Monza', 'Silverstone', 'Spa', 'Suzuka', 'Barcelona', 'Catalunya', 'Zandvoort', 'Austin', 'COTA', 'Melbourne', 'Albert Park']
Monaco geometry: None


### 1c. Load everything at once + generate a text description

In [8]:
# load_all_track_data wraps both loaders
track = load_all_track_data('Silverstone')

print('Keys:', list(track.keys()))
print('Metadata name:', track['metadata']['name'])
print('Geometry rows:', len(track['geometry']) if track['geometry'] is not None else 'N/A')


Loading track data for: Silverstone
  Metadata: Silverstone Circuit, 5891m, alt=196m, 135 GPS points
  Geometry: 1178 centerline points with track widths
Keys: ['metadata', 'geometry']
Metadata name: Silverstone Circuit
Geometry rows: 1178


In [9]:
# describe_circuit() generates a plain-text chunk ready for embedding
# This is exactly what will go into our vector store later
print('--- RAG document chunk (what we will embed) ---')
print()
print(describe_circuit('Monza'))
print()
print(describe_circuit('Monaco'))

--- RAG document chunk (what we will embed) ---


Loading track data for: Monza
  Metadata: Autodromo Nazionale Monza, 5793m, alt=142m, 125 GPS points
  Geometry: 1159 centerline points with track widths
Circuit: Autodromo Nazionale Monza
Location: Monza
Track length: 5793m (5.793 km)
Altitude: 142m above sea level
First Formula 1 Grand Prix: 1950
Circuit opened: 1922
Average track width: 4.7m


Loading track data for: Monaco
  Metadata: Circuit de Monaco, 3337m, alt=47m, 160 GPS points
  Note: Monaco not in TUMFTM database. Available: ['Monza', 'Silverstone', 'Spa', 'Suzuka', 'Barcelona', 'Catalunya', 'Zandvoort', 'Austin', 'COTA', 'Melbourne', 'Albert Park']
  Geometry: not available (GPS outline only)
Circuit: Circuit de Monaco
Location: Monaco
Track length: 3337m (3.337 km)
Altitude: 47m above sea level
First Formula 1 Grand Prix: 1929
Circuit opened: 1929


---
## Part 2: FastF1 Qualifying Session

Requires internet access to `livetiming.formula1.com`.  
First run will be slow (downloading data). Subsequent runs use the local cache.

In [10]:
from load_qualifying import load_qualifying_session, summarize_session

print('FastF1 functions imported.')

FastF1 functions imported.


### 2a. Load a session

In [11]:
# Load Monza 2024 qualifying — takes ~30s on first run
data = load_qualifying_session(year=2024, circuit='Monza')

# What did we get?
print('Keys:', list(data.keys()))
print()
print('Session info:', data['session_info'])

Loading 2024 Monza Qualifying...


core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '63', '16', '55', '44', '1', '11', '23', '27', '14', '3', '20', '10', '31', '22', '18', '43', '77', '24']
/Users/ZaneQureshi/Desktop/ML/f1-analytics-/RAG_data_layers/F1 Project Thoughts/load_qualifying.py:59: Futur

  Loaded 20 drivers, 176 total laps
  Telemetry loaded for: ['NOR', 'PIA', 'RUS', 'LEC', 'SAI', 'HAM', 'VER', 'PER', 'ALB', 'HUL', 'ALO', 'RIC', 'MAG', 'GAS', 'OCO', 'TSU', 'STR', 'COL', 'BOT', 'ZHO']
Keys: ['session_info', 'driver_fastest_laps', 'all_laps', 'weather', 'telemetry']

Session info: {'year': 2024, 'circuit': 'Monza', 'event_name': 'Italian Grand Prix', 'location': 'Monza', 'country': 'Italy', 'date': '2024-08-31 14:00:00'}


In [12]:
# Pretty-print the full qualifying order
summarize_session(data)


Italian Grand Prix 2024 - Qualifying
Circuit: Monza, Italy

Qualifying Order (fastest laps):
  P 1  NOR  (McLaren)  00:01:19.327000
  P 2  PIA  (McLaren)  00:01:19.436000
  P 3  RUS  (Mercedes)  00:01:19.440000
  P 4  LEC  (Ferrari)  00:01:19.461000
  P 5  SAI  (Ferrari)  00:01:19.467000
  P 6  HAM  (Mercedes)  00:01:19.513000
  P 7  VER  (Red Bull Racing)  00:01:19.662000
  P 8  PER  (Red Bull Racing)  00:01:20.062000
  P 9  ALB  (Williams)  00:01:20.299000
  P10  HUL  (Haas F1 Team)  00:01:20.339000
  P11  ALO  (Aston Martin)  00:01:20.421000
  P12  RIC  (RB)  00:01:20.479000
  P13  MAG  (Haas F1 Team)  00:01:20.698000
  P14  GAS  (Alpine)  00:01:20.738000
  P15  OCO  (Alpine)  00:01:20.764000
  P16  TSU  (RB)  00:01:20.945000
  P17  STR  (Aston Martin)  00:01:21.013000
  P18  COL  (Williams)  00:01:21.061000
  P19  BOT  (Kick Sauber)  00:01:21.101000
  P20  ZHO  (Kick Sauber)  00:01:21.445000

Weather avg: Track 48.7°C / Air 33.4°C


### 2b. Explore driver fastest laps

In [13]:
fastest = data['driver_fastest_laps']

print('Shape:', fastest.shape)
print('Columns:', list(fastest.columns))
print()
fastest[['Driver', 'Team', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
         'SpeedFL', 'Compound', 'QualifyingPosition']]

Shape: (20, 18)
Columns: ['Driver', 'DriverNumber', 'Team', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'Compound', 'TyreLife', 'FreshTyre', 'LapStartTime', 'LapNumber', 'IsPersonalBest', 'QualifyingPosition']



,Driver,Team,LapTime,Sector1Time,Sector2Time,Sector3Time,SpeedFL,Compound,QualifyingPosition
0,NOR,McLaren,0 days 00:01:19.327000,0 days 00:00:26.492000,0 days 00:00:26.579000,0 days 00:00:26.256000,315.000,SOFT,1
1,PIA,McLaren,0 days 00:01:19.436000,0 days 00:00:26.407000,0 days 00:00:26.643000,0 days 00:00:26.386000,315.000,SOFT,2
2,RUS,Mercedes,0 days 00:01:19.440000,0 days 00:00:26.296000,0 days 00:00:26.844000,0 days 00:00:26.300000,315.000,SOFT,3
3,LEC,Ferrari,0 days 00:01:19.461000,0 days 00:00:26.373000,0 days 00:00:26.732000,0 days 00:00:26.356000,318.000,SOFT,4
4,SAI,Ferrari,0 days 00:01:19.467000,0 days 00:00:26.223000,0 days 00:00:26.823000,0 days 00:00:26.421000,313.000,SOFT,5
5,HAM,Mercedes,0 days 00:01:19.513000,0 days 00:00:26.428000,0 days 00:00:26.672000,0 days 00:00:26.413000,316.000,SOFT,6
6,VER,Red Bull Racing,0 days 00:01:19.662000,0 days 00:00:26.389000,0 days 00:00:26.656000,0 days 00:00:26.617000,317.000,SOFT,7
7,PER,Red Bull Racing,0 days 00:01:20.062000,0 days 00:00:26.488000,0 days 00:00:27.042000,0 days 00:00:26.532000,313.000,SOFT,8
8,ALB,Williams,0 days 00:01:20.299000,0 days 00:00:26.395000,0 days 00:00:27.120000,0 days 00:00:26.784000,314.000,SOFT,9
9,HUL,Haas F1 Team,0 days 00:01:20.339000,0 days 00:00:26.479000,0 days 00:00:27.074000,0 days 00:00:26.786000,317.000,SOFT,10


In [14]:
# Who was fastest in each sector?
for sector in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
    best_row = fastest.dropna(subset=[sector]).sort_values(sector).iloc[0]
    print(f'{sector}: {best_row["Driver"]} ({best_row["Team"]}) — {best_row[sector]}')

Sector1Time: SAI (Ferrari) — 0 days 00:00:26.223000
Sector2Time: NOR (McLaren) — 0 days 00:00:26.579000
Sector3Time: NOR (McLaren) — 0 days 00:00:26.256000


In [15]:
# Compare top 5 to pole time
pole_time = fastest.iloc[0]['LapTime']
top5 = fastest.head(5).copy()
top5['delta_to_pole'] = top5['LapTime'] - pole_time

top5[['Driver', 'Team', 'LapTime', 'delta_to_pole', 'QualifyingPosition']]

,Driver,Team,LapTime,delta_to_pole,QualifyingPosition
0,NOR,McLaren,0 days 00:01:19.327000,0 days 00:00:00,1
1,PIA,McLaren,0 days 00:01:19.436000,0 days 00:00:00.109000,2
2,RUS,Mercedes,0 days 00:01:19.440000,0 days 00:00:00.113000,3
3,LEC,Ferrari,0 days 00:01:19.461000,0 days 00:00:00.134000,4
4,SAI,Ferrari,0 days 00:01:19.467000,0 days 00:00:00.140000,5


### 2c. Explore telemetry for a single driver

In [16]:
telemetry = data['telemetry']

print('Drivers with telemetry:', list(telemetry.keys()))

# Pick the pole sitter
pole_driver = fastest.iloc[0]['Driver']
tel = telemetry[pole_driver]

print(f'\nTelemetry for {pole_driver}:')
print('Shape:', tel.shape)
print('Columns:', list(tel.columns))

Drivers with telemetry: ['NOR', 'PIA', 'RUS', 'LEC', 'SAI', 'HAM', 'VER', 'PER', 'ALB', 'HUL', 'ALO', 'RIC', 'MAG', 'GAS', 'OCO', 'TSU', 'STR', 'COL', 'BOT', 'ZHO']

Telemetry for NOR:
Shape: (616, 19)
Columns: ['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z', 'Driver']


In [17]:
from load_qualifying import load_qualifying_session, summarize_session

print('Qualifying data functions imported.')

Qualifying data functions imported.


In [18]:
# Compare max speed between top 3 qualifiers
print('Max speed comparison (top 3):')
for driver in fastest.head(3)['Driver']:
    if driver in telemetry and 'Speed' in telemetry[driver].columns:
        max_speed = telemetry[driver]['Speed'].max()
        pos = fastest[fastest['Driver'] == driver]['QualifyingPosition'].values[0]
        print(f'  P{pos} {driver}: {max_speed:.1f} km/h')

Max speed comparison (top 3):
  P1 NOR: 347.0 km/h
  P2 PIA: 346.0 km/h
  P3 RUS: 348.0 km/h


### 2d. Explore all laps (not just fastest)

In [19]:
all_laps = data['all_laps']

print('Total laps:', len(all_laps))
print('Columns:', list(all_laps.columns))
print()

# How many laps did each driver do?
lap_counts = all_laps.groupby('Driver').size().sort_values(ascending=False)
print('Laps per driver:')
print(lap_counts)

Total laps: 176
Columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']

Laps per driver:
Driver
LEC    14
PIA    12
SAI    12
ALB    11
VER    11
RUS    11
HAM    11
GAS    11
OCO    11
HUL    11
PER    10
RIC     8
MAG     7
NOR     7
ALO     7
ZHO     6
COL     4
BOT     4
TSU     4
STR     4
dtype: int64


In [20]:
# Track evolution: did lap times improve across the session?
# Look at the overall fastest lap each minute of the session
all_laps_sorted = all_laps.sort_values('LapStartTime')

# Just show the progression of personal bests for pole driver
pole_laps = all_laps[all_laps['Driver'] == pole_driver].sort_values('LapStartTime')
print(f'Lap progression for {pole_driver}:')
pole_laps[['LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Compound']]

Lap progression for NOR:


,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound
1,2.000,0 days 00:01:19.911000,0 days 00:00:26.484000,0 days 00:00:26.898000,0 days 00:00:26.529000,SOFT
2,3.000,0 days 00:01:51.021000,0 days 00:00:32.371000,0 days 00:00:38.291000,0 days 00:00:40.359000,SOFT
4,5.000,0 days 00:01:19.727000,0 days 00:00:26.395000,0 days 00:00:26.865000,0 days 00:00:26.467000,SOFT
5,6.000,0 days 00:01:49.446000,0 days 00:00:31.980000,0 days 00:00:37.478000,0 days 00:00:39.988000,SOFT
7,8.000,0 days 00:01:19.401000,0 days 00:00:26.400000,0 days 00:00:26.748000,0 days 00:00:26.253000,SOFT
8,9.000,0 days 00:01:47.128000,0 days 00:00:30.949000,0 days 00:00:34.968000,0 days 00:00:41.211000,SOFT
10,11.000,0 days 00:01:19.327000,0 days 00:00:26.492000,0 days 00:00:26.579000,0 days 00:00:26.256000,SOFT


---
## Part 3: OpenF1 — Weather + Race Control

Requires internet access to `api.openf1.org`.  
Data available from 2023 onwards, no authentication needed.

In [21]:
from load_openf1 import (
    load_openf1_qualifying_context,
    summarize_openf1_context,
    get_session_key,

    )

print('OpenF1 functions imported.')

OpenF1 functions imported.


### 3a. Resolve a session key

In [22]:
# OpenF1 uses a session_key integer to identify each session
# get_session_key() resolves year + circuit_short_name -> session_key
session_key, session_info = get_session_key(year=2024, circuit_short_name='monza')

print('Session key:', session_key)
print('Session info keys:', list(session_info.keys()))
print()
for k, v in session_info.items():
    print(f'  {k}: {v}')

Found session: Qualifying @ Monza (2024-08-31) | session_key=9586
Session key: 9586
Session info keys: ['session_key', 'session_type', 'session_name', 'date_start', 'date_end', 'meeting_key', 'circuit_key', 'circuit_short_name', 'country_key', 'country_code', 'country_name', 'location', 'gmt_offset', 'year', 'is_cancelled']

  session_key: 9586
  session_type: Qualifying
  session_name: Qualifying
  date_start: 2024-08-31T14:00:00+00:00
  date_end: 2024-08-31T15:00:00+00:00
  meeting_key: 1244
  circuit_key: 39
  circuit_short_name: Monza
  country_key: 13
  country_code: ITA
  country_name: Italy
  location: Monza
  gmt_offset: 02:00:00
  year: 2024
  is_cancelled: False


### 3b. Load all OpenF1 context at once

In [23]:
# Load everything for Monza 2024 qualifying
ctx = load_openf1_qualifying_context(year=2024, circuit_short_name='monza')

print('Keys:', list(ctx.keys()))


Loading OpenF1 data: 2024 monza Qualifying
Found session: Qualifying @ Monza (2024-08-31) | session_key=9586
  Weather: 79 snapshots
  Race control messages: 36
  Messages: ['GREEN LIGHT - PIT EXIT OPEN', 'SESSION STARTED', 'CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TURN 2 LAP 3 16:05:33', 'CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 16:10:08 (PIT)', 'INCIDENT INVOLVING CAR 81 (PIA) NOTED - UNSAFE RELEASE', 'FIA STEWARDS: INCIDENT INVOLVING CAR 81 (PIA) WILL BE INVESTIGATED AFTER THE SESSION - UNSAFE RELEASE', 'YELLOW IN TRACK SECTOR 15', 'CLEAR IN TRACK SECTOR 15', 'CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TURN 7 LAP 6 16:16:37', 'CHEQUERED FLAG', 'SESSION FINISHED', 'FIRST CAR TO TAKE THE FLAG - CAR 81 (PIA)', 'CAR 20 (MAG) LAP DELETED - TRACK LIMITS AT TURN 11 LAP 6 16:17:51', 'CAR 43 (COL) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 7 16:17:39 (PIT)', 'CAR 27 (HUL) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 7 16:17:42', 'FIA STEWARDS: Q1 INCIDENT INVO

### 3c. Explore weather data

In [24]:
weather = ctx['weather']

print('Weather shape:', weather.shape)
print('Columns:', list(weather.columns))
print()
weather.head(10)

Weather shape: (79, 12)
Columns: ['date', 'session_key', 'meeting_key', 'track_temperature', 'wind_speed', 'rainfall', 'humidity', 'pressure', 'air_temperature', 'wind_direction', 'air_temp', 'track_temp']



,date,session_key,meeting_key,track_temperature,wind_speed,rainfall,humidity,pressure,air_temperature,wind_direction,air_temp,track_temp
0,2024-08-31 13:49:03.993000+00:00,9586,1244,48.400,1.200,0,37.000,996.300,33.100,243,33.100,48.400
1,2024-08-31 13:50:03.995000+00:00,9586,1244,48.400,0.700,0,37.000,996.200,33.100,186,33.100,48.400
2,2024-08-31 13:51:03.990000+00:00,9586,1244,48.400,0.600,0,37.000,996.200,33.000,234,33.000,48.400
3,2024-08-31 13:52:04.014000+00:00,9586,1244,49.900,0.600,0,37.000,996.200,33.100,178,33.100,49.900
4,2024-08-31 13:53:03.985000+00:00,9586,1244,50.100,1.100,0,37.000,996.100,33.200,186,33.200,50.100
5,2024-08-31 13:54:03.988000+00:00,9586,1244,50.200,2.300,0,37.000,996.100,33.300,200,33.300,50.200
6,2024-08-31 13:55:03.996000+00:00,9586,1244,50.200,1.600,0,37.000,996.200,33.300,230,33.300,50.200
7,2024-08-31 13:56:03.992000+00:00,9586,1244,50.100,1.400,0,37.000,996.100,33.300,247,33.300,50.100
8,2024-08-31 13:57:03.989000+00:00,9586,1244,50.000,1.300,0,37.000,996.100,33.500,197,33.500,50.000
9,2024-08-31 13:58:03.999000+00:00,9586,1244,50.100,1.100,0,37.000,996.100,33.400,201,33.400,50.100


In [25]:
# Did track temperature change across the session? (affects lap time potential)
if 'track_temp' in weather.columns:
    print('Track temperature across session:')
    print(f'  Start:  {weather["track_temp"].iloc[0]:.1f}°C')
    print(f'  End:    {weather["track_temp"].iloc[-1]:.1f}°C')
    print(f'  Min:    {weather["track_temp"].min():.1f}°C')
    print(f'  Max:    {weather["track_temp"].max():.1f}°C')
    print(f'  Change: {weather["track_temp"].iloc[-1] - weather["track_temp"].iloc[0]:+.1f}°C')

if 'rainfall' in weather.columns:
    print(f'\nRainfall detected: {weather["rainfall"].any()}')

Track temperature across session:
  Start:  48.4°C
  End:    46.4°C
  Min:    46.2°C
  Max:    51.0°C
  Change: -2.0°C

Rainfall detected: False


### 3d. Race control messages — the most important OpenF1 data for your RAG system

In [26]:
rc = ctx['race_control']

print('Race control shape:', rc.shape)
print('Columns:', list(rc.columns))
print()
rc

Race control shape: (36, 11)
Columns: ['meeting_key', 'session_key', 'date', 'driver_number', 'lap_number', 'category', 'flag', 'scope', 'sector', 'qualifying_phase', 'message']



,meeting_key,session_key,date,driver_number,lap_number,category,flag,scope,sector,qualifying_phase,message
0,1244,9586,2024-08-31 14:00:00+00:00,None,None,Flag,GREEN,Track,NaN,1.000,GREEN LIGHT - PIT EXIT OPEN
1,1244,9586,2024-08-31 14:00:00.070000+00:00,None,None,SessionStatus,None,None,NaN,1.000,SESSION STARTED
2,1244,9586,2024-08-31 14:07:15+00:00,None,None,Other,None,None,NaN,NaN,CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TUR...
3,1244,9586,2024-08-31 14:13:19+00:00,None,None,Other,None,None,NaN,NaN,CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 ...
4,1244,9586,2024-08-31 14:14:43+00:00,None,None,Other,None,None,NaN,NaN,INCIDENT INVOLVING CAR 81 (PIA) NOTED - UNSAFE RELEASE
5,1244,9586,2024-08-31 14:15:47+00:00,None,None,Other,None,None,NaN,NaN,FIA STEWARDS: INCIDENT INVOLVING CAR 81 (PIA) WILL BE IN...
6,1244,9586,2024-08-31 14:17:19+00:00,None,None,Flag,YELLOW,Sector,15.000,1.000,YELLOW IN TRACK SECTOR 15
7,1244,9586,2024-08-31 14:17:28+00:00,None,None,Flag,CLEAR,Sector,15.000,1.000,CLEAR IN TRACK SECTOR 15
8,1244,9586,2024-08-31 14:17:29+00:00,None,None,Other,None,None,NaN,NaN,CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TUR...
9,1244,9586,2024-08-31 14:18:00+00:00,None,None,Flag,CHEQUERED,Track,NaN,1.000,CHEQUERED FLAG


In [27]:
# Flag events only — these are what cause slow laps
if 'flag' in rc.columns:
    flags = rc[rc['flag'].notna() & (rc['flag'] != '')]
    print(f'Flag events: {len(flags)}')
    print()
    print(flags[['date', 'flag', 'message', 'category']])

Flag events: 8

                        date       flag                      message category
0  2024-08-31 14:00:00+00:00      GREEN  GREEN LIGHT - PIT EXIT OPEN     Flag
6  2024-08-31 14:17:19+00:00     YELLOW    YELLOW IN TRACK SECTOR 15     Flag
7  2024-08-31 14:17:28+00:00      CLEAR     CLEAR IN TRACK SECTOR 15     Flag
9  2024-08-31 14:18:00+00:00  CHEQUERED               CHEQUERED FLAG     Flag
19 2024-08-31 14:29:00+00:00      GREEN  GREEN LIGHT - PIT EXIT OPEN     Flag
21 2024-08-31 14:44:00+00:00  CHEQUERED               CHEQUERED FLAG     Flag
27 2024-08-31 14:52:00+00:00      GREEN  GREEN LIGHT - PIT EXIT OPEN     Flag
31 2024-08-31 15:04:00+00:00  CHEQUERED               CHEQUERED FLAG     Flag


In [28]:
# Lap time deletions — another common RAG answer for "why was lap X slow/invalid"
if 'message' in rc.columns:
    deletions = rc[rc['message'].str.contains('DELETED|deletion|track limits', case=False, na=False)]
    print(f'Lap deletion messages: {len(deletions)}')
    if len(deletions) > 0:
        print(deletions[['date', 'message']])

Lap deletion messages: 6
                        date  \
2  2024-08-31 14:07:15+00:00   
3  2024-08-31 14:13:19+00:00   
8  2024-08-31 14:17:29+00:00   
12 2024-08-31 14:18:19+00:00   
13 2024-08-31 14:19:11+00:00   
14 2024-08-31 14:19:25+00:00   

                                                        message  
2   CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TUR...  
3   CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 ...  
8   CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TUR...  
12  CAR 20 (MAG) LAP DELETED - TRACK LIMITS AT TURN 11 LAP 6...  
13  CAR 43 (COL) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 7 ...  
14  CAR 27 (HUL) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 7 ...  


### 3e. Driver metadata

In [29]:
drivers = ctx['drivers']
print('Columns:', list(drivers.columns))
print()

# Show key fields
keep = [c for c in ['full_name', 'name_acronym', 'team_name', 'driver_number', 'country_code'] 
        if c in drivers.columns]
drivers[keep].sort_values('driver_number' if 'driver_number' in keep else keep[0])

Columns: ['meeting_key', 'session_key', 'driver_number', 'broadcast_name', 'full_name', 'name_acronym', 'team_name', 'team_colour', 'first_name', 'last_name', 'headshot_url', 'country_code']



,full_name,name_acronym,team_name,driver_number,country_code
0,Max VERSTAPPEN,VER,Red Bull Racing,1,NED
1,Daniel RICCIARDO,RIC,RB,3,AUS
2,Lando NORRIS,NOR,McLaren,4,GBR
3,Pierre GASLY,GAS,Alpine,10,FRA
4,Sergio PEREZ,PER,Red Bull Racing,11,MEX
5,Fernando ALONSO,ALO,Aston Martin,14,ESP
6,Charles LECLERC,LEC,Ferrari,16,MON
7,Lance STROLL,STR,Aston Martin,18,CAN
8,Kevin MAGNUSSEN,MAG,Haas F1 Team,20,DEN
9,Yuki TSUNODA,TSU,RB,22,JPN


### 3f. Full summary

In [30]:
summarize_openf1_context(ctx)


Weather Summary:
  air_temp: min=33.0, max=33.9, mean=33.4
  track_temp: min=46.2, max=51.0, mean=48.7
  humidity: min=35.0, max=39.0, mean=36.8
  rainfall: min=0.0, max=0.0, mean=0.0

Race Control (36 messages):
  [Flag] GREEN — GREEN LIGHT - PIT EXIT OPEN
  [SessionStatus] None — SESSION STARTED
  [Other] None — CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TURN 2 LAP 3 16:05:33
  [Other] None — CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 16:10:08 (PIT)
  [Other] None — INCIDENT INVOLVING CAR 81 (PIA) NOTED - UNSAFE RELEASE
  [Other] None — FIA STEWARDS: INCIDENT INVOLVING CAR 81 (PIA) WILL BE INVESTIGATED AFTER THE SESSION - UNSAFE RELEASE
  [Flag] YELLOW — YELLOW IN TRACK SECTOR 15
  [Flag] CLEAR — CLEAR IN TRACK SECTOR 15
  [Other] None — CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TURN 7 LAP 6 16:16:37
  [Flag] CHEQUERED — CHEQUERED FLAG
  [SessionStatus] None — SESSION FINISHED
  [Other] None — FIRST CAR TO TAKE THE FLAG - CAR 81 (PIA)
  [Other] None

---
## Part 4: Combining All Three Sources

This is what the chunking layer will eventually do — bring all three data layers
together so the LLM has full context to answer qualifying questions.

In [31]:
# Load all three for the same circuit
YEAR    = 2024
CIRCUIT = 'Monza'

# 1. Track geometry
track = load_all_track_data(CIRCUIT)

# 2. FastF1 session (comment out if no F1 API access)
session_data = load_qualifying_session(year=YEAR, circuit=CIRCUIT)

# 3. OpenF1 context (comment out if no OpenF1 API access)
openf1_ctx = load_openf1_qualifying_context(year=YEAR, circuit_short_name=CIRCUIT)

print('\nAll three sources loaded.')


Loading track data for: Monza
  Metadata: Autodromo Nazionale Monza, 5793m, alt=142m, 125 GPS points


core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Geometry: 1159 centerline points with track widths
Loading 2024 Monza Qualifying...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '63', '16', '55', '44', '1', '11', '23', '27', '14', '3', '20', '10', '31', '22', '18', '43', '77', '24']
/Users/ZaneQureshi/Desktop/ML/f1-analytics-/RAG_data_layers/F1 Project Thoughts/load_qualifying.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Ei

  Loaded 20 drivers, 176 total laps
  Telemetry loaded for: ['NOR', 'PIA', 'RUS', 'LEC', 'SAI', 'HAM', 'VER', 'PER', 'ALB', 'HUL', 'ALO', 'RIC', 'MAG', 'GAS', 'OCO', 'TSU', 'STR', 'COL', 'BOT', 'ZHO']

Loading OpenF1 data: 2024 Monza Qualifying
Found session: Qualifying @ Monza (2024-08-31) | session_key=9586
  Weather: 79 snapshots
  Race control messages: 36
  Messages: ['GREEN LIGHT - PIT EXIT OPEN', 'SESSION STARTED', 'CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TURN 2 LAP 3 16:05:33', 'CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 16:10:08 (PIT)', 'INCIDENT INVOLVING CAR 81 (PIA) NOTED - UNSAFE RELEASE', 'FIA STEWARDS: INCIDENT INVOLVING CAR 81 (PIA) WILL BE INVESTIGATED AFTER THE SESSION - UNSAFE RELEASE', 'YELLOW IN TRACK SECTOR 15', 'CLEAR IN TRACK SECTOR 15', 'CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TURN 7 LAP 6 16:16:37', 'CHEQUERED FLAG', 'SESSION FINISHED', 'FIRST CAR TO TAKE THE FLAG - CAR 81 (PIA)', 'CAR 20 (MAG) LAP DELETED - TRACK LIMITS 

### 4a. Parameterized session selection

Set the year, circuit, and session details here, then rerun the loading and retrieval cells to switch away from Monza.

In [32]:
SESSION_YEAR = 2024
SESSION_CIRCUIT = 'Monza'
SESSION_SHORT_NAME = 'monza'
SESSION_LABEL = 'Qualifying'

print('Configured session:')
print('  Year:', SESSION_YEAR)
print('  Circuit:', SESSION_CIRCUIT)
print('  Short name:', SESSION_SHORT_NAME)
print('  Session:', SESSION_LABEL)

Configured session:
  Year: 2024
  Circuit: Monza
  Short name: monza
  Session: Qualifying


In [33]:
# Reload the selected session and rebuild the corpus for the chosen circuit/year.
# If you change SESSION_YEAR or SESSION_CIRCUIT above, rerun this cell and everything below it.
track = load_all_track_data(SESSION_CIRCUIT)
session_data = load_qualifying_session(year=SESSION_YEAR, circuit=SESSION_CIRCUIT)
openf1_ctx = load_openf1_qualifying_context(year=SESSION_YEAR, circuit_short_name=SESSION_SHORT_NAME)

print('\nSelected session loaded.')
print('Track:', track['metadata']['name'])
print('FastF1 event:', session_data['session_info']['event_name'])
print('OpenF1 session key:', openf1_ctx['session_info']['session_key'])


Loading track data for: Monza
  Metadata: Autodromo Nazionale Monza, 5793m, alt=142m, 125 GPS points


core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


  Geometry: 1159 centerline points with track widths
Loading 2024 Monza Qualifying...


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '63', '16', '55', '44', '1', '11', '23', '27', '14', '3', '20', '10', '31', '22', '18', '43', '77', '24']
/Users/ZaneQureshi/Desktop/ML/f1-analytics-/RAG_data_layers/F1 Project Thoughts/load_qualifying.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grp: grp.pick_fastest())
/Users/ZaneQureshi/Desktop/ML/f1-analytics-/venv/lib/python3.13/site-packages/fastf1/core.py:3175: FutureWarning: p

  Loaded 20 drivers, 176 total laps
  Telemetry loaded for: ['NOR', 'PIA', 'RUS', 'LEC', 'SAI', 'HAM', 'VER', 'PER', 'ALB', 'HUL', 'ALO', 'RIC', 'MAG', 'GAS', 'OCO', 'TSU', 'STR', 'COL', 'BOT', 'ZHO']

Loading OpenF1 data: 2024 monza Qualifying
Found session: Qualifying @ Monza (2024-08-31) | session_key=9586
  Weather: 79 snapshots
  Race control messages: 36
  Messages: ['GREEN LIGHT - PIT EXIT OPEN', 'SESSION STARTED', 'CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TURN 2 LAP 3 16:05:33', 'CAR 44 (HAM) LAP DELETED - TRACK LIMITS AT TURN 2 LAP 4 16:10:08 (PIT)', 'INCIDENT INVOLVING CAR 81 (PIA) NOTED - UNSAFE RELEASE', 'FIA STEWARDS: INCIDENT INVOLVING CAR 81 (PIA) WILL BE INVESTIGATED AFTER THE SESSION - UNSAFE RELEASE', 'YELLOW IN TRACK SECTOR 15', 'CLEAR IN TRACK SECTOR 15', 'CAR 43 (COL) TIME 1:22.531 DELETED - TRACK LIMITS AT TURN 7 LAP 6 16:16:37', 'CHEQUERED FLAG', 'SESSION FINISHED', 'FIRST CAR TO TAKE THE FLAG - CAR 81 (PIA)', 'CAR 20 (MAG) LAP DELETED - TRACK LIMITS 

In [34]:
# Combine into a context snapshot — prototype of what a RAG chunk will look like
# for a specific driver's qualifying lap

def build_driver_lap_context(driver_code: str) -> str:
    """
    Assemble a plain-text context block for a driver's qualifying lap.
    This is a prototype of the chunks we'll embed in the vector store.
    """
    info    = session_data['session_info']
    fastest = session_data['driver_fastest_laps']
    weather = openf1_ctx.get('weather', pd.DataFrame())
    rc      = openf1_ctx.get('race_control', pd.DataFrame())
    track_meta = track['metadata']

    # Driver row
    row = fastest[fastest['Driver'] == driver_code]
    if row.empty:
        return f'No data found for driver {driver_code}'
    row = row.iloc[0]

    # Weather averages from OpenF1, with aliases added by the loader
    track_temp_col = 'track_temp' if 'track_temp' in weather.columns else 'track_temperature'
    air_temp_col = 'air_temp' if 'air_temp' in weather.columns else 'air_temperature'
    rain_col = 'rainfall' if 'rainfall' in weather.columns else None
    wind_col = 'wind_speed' if 'wind_speed' in weather.columns else None

    avg_track_temp = weather[track_temp_col].mean() if track_temp_col in weather.columns else 'N/A'
    avg_air_temp = weather[air_temp_col].mean() if air_temp_col in weather.columns else 'N/A'
    avg_wind_speed = weather[wind_col].mean() if wind_col in weather.columns else 'N/A'
    rainfall = bool(weather[rain_col].any()) if rain_col and rain_col in weather.columns else 'N/A'

    # Race control flags
    flags = rc[rc['flag'].notna()] if 'flag' in rc.columns else pd.DataFrame()
    flag_summary = ', '.join(flags['flag'].unique().tolist()) if len(flags) else 'None'
    messages = rc[rc['message'].notna()] if 'message' in rc.columns else pd.DataFrame()
    message_summary = '; '.join(messages['message'].drop_duplicates().head(3).tolist()) if len(messages) else 'None'

    lines = [
        f"Event: {info['event_name']} {info['year']} — Qualifying",
        f"Circuit: {track_meta['name']} ({track_meta['length_m']}m, alt {track_meta['altitude_m']}m)",
        f"Driver: {row['Driver']} | Team: {row.get('Team', 'N/A')} | P{int(row['QualifyingPosition'])}",
        f"Lap Time: {row['LapTime']}",
        f"Sectors: S1={row['Sector1Time']} | S2={row['Sector2Time']} | S3={row['Sector3Time']}",
        f"Speed Trap: FL={row.get('SpeedFL', 'N/A')} km/h | ST={row.get('SpeedST', 'N/A')} km/h",
        f"Tyre: {row.get('Compound', 'N/A')}",
        f"OpenF1 Weather: Air {avg_air_temp:.1f}°C | Track {avg_track_temp:.1f}°C | Wind {avg_wind_speed} m/s | Rain: {rainfall}",
        f"OpenF1 race control flags: {flag_summary}",
        f"OpenF1 messages: {message_summary}",
    ]

    return '\n'.join(lines)


# Generate context blocks for top 3 qualifiers
for driver in session_data['driver_fastest_laps'].head(3)['Driver']:
    print('=' * 55)
    print(build_driver_lap_context(driver))
    print()

Event: Italian Grand Prix 2024 — Qualifying
Circuit: Autodromo Nazionale Monza (5793m, alt 142m)
Driver: NOR | Team: McLaren | P1
Lap Time: 0 days 00:01:19.327000
Sectors: S1=0 days 00:00:26.492000 | S2=0 days 00:00:26.579000 | S3=0 days 00:00:26.256000
Speed Trap: FL=315.0 km/h | ST=346.0 km/h
Tyre: SOFT
OpenF1 Weather: Air 33.4°C | Track 48.7°C | Wind 1.3708860759493673 m/s | Rain: False
OpenF1 race control flags: GREEN, YELLOW, CLEAR, CHEQUERED
OpenF1 messages: GREEN LIGHT - PIT EXIT OPEN; SESSION STARTED; CAR 81 (PIA) TIME 1:50.311 DELETED - TRACK LIMITS AT TURN 2 LAP 3 16:05:33

Event: Italian Grand Prix 2024 — Qualifying
Circuit: Autodromo Nazionale Monza (5793m, alt 142m)
Driver: PIA | Team: McLaren | P2
Lap Time: 0 days 00:01:19.436000
Sectors: S1=0 days 00:00:26.407000 | S2=0 days 00:00:26.643000 | S3=0 days 00:00:26.386000
Speed Trap: FL=315.0 km/h | ST=345.0 km/h
Tyre: SOFT
OpenF1 Weather: Air 33.4°C | Track 48.7°C | Wind 1.3708860759493673 m/s | Rain: False
OpenF1 race cont

---
## Part 5: Embeddings + ChromaDB

This section turns the chunk objects into vectors, stores them in a local ChromaDB collection, and runs retrieval checks with metadata filters.

### Embedding model: `all-MiniLM-L6-v2` via `fastembed`

We use [fastembed](https://github.com/qdrant/fastembed) rather than sentence-transformers or an API-based service:

| | fastembed (this notebook) | sentence-transformers | API-based (e.g. Voyage, OpenAI) |
|---|---|---|---|
| **Backend** | ONNX Runtime (~50 MB) | PyTorch (~2 GB) | Remote API |
| **Setup** | `pip install fastembed` | Requires torch | API key + account |
| **Cost** | Free | Free | Pay per token |
| **Installs on any platform** | Yes | Platform-specific wheels | N/A |
| **Quality** | Semantically meaningful | Same model | Marginally better |

fastembed is purpose-built for RAG inference — same `all-MiniLM-L6-v2` model, quantized to ONNX, with no training framework overhead. 
The `embed_fn` interface in `rag_retrieval.py` is provider-agnostic: `make_embed_fn()` in that file shows how to swap models.

In [35]:
import json
from pathlib import Path

import chromadb
import numpy as np
from IPython.display import display

from events_and_chunking import build_all_chunks, chunk_summary, validate_chunks

print('Embedding stack imported.')

chunks = build_all_chunks(
    fastf1_data=session_data,
    openf1_data=openf1_ctx,
    track_data=track,
    session_info=session_data['session_info'],
    head_to_head_top_n=5,
    verbose=False,
)

print(chunk_summary(chunks))
warnings = validate_chunks(chunks)
print('\nValidation warnings:', warnings if warnings else 'None')

chunk_preview = pd.DataFrame([
    {
        'chunk_type': chunk.chunk_type,
        'driver': chunk.metadata.get('driver'),
        'event_id': chunk.metadata.get('event_id'),
        'source': chunk.source,
        'text_preview': chunk.text[:120].replace('\n', ' '),
    }
    for chunk in chunks
])

display(chunk_preview.head(10))

Embedding stack imported.
Total chunks: 857

  circuit_overview: 4
  driver_lap_summary: 20
  driver_sector: 60
  driver_telemetry_zone: 60
  head_to_head: 10
  head_to_head_event: 306
  race_control_event: 36
  session_pace_evolution: 20
  session_weather: 1
  telemetry_corner_event: 180
  telemetry_straight_event: 160

Validation warnings: None


,chunk_type,driver,event_id,source,text_preview
0,circuit_overview,None,None,bacinger+tumftm,Circuit Overview: Autodromo Nazionale Monza Location: Mo...
1,circuit_overview,None,None,tumftm,Circuit: Autodromo Nazionale Monza — Geometry Zone 1 of ...
2,circuit_overview,None,None,tumftm,Circuit: Autodromo Nazionale Monza — Geometry Zone 2 of ...
3,circuit_overview,None,None,tumftm,Circuit: Autodromo Nazionale Monza — Geometry Zone 3 of ...
4,session_weather,None,None,openf1,Weather Conditions: Monza 2024 Qualifying Track temperat...
5,race_control_event,None,None,openf1,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
6,race_control_event,None,None,openf1,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
7,race_control_event,None,None,openf1,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
8,race_control_event,None,None,openf1,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
9,race_control_event,None,None,openf1,Race Control Event — Monza 2024 Qualifying Time: 2024-08...


In [36]:
from fastembed import TextEmbedding

# Downloads ~40 MB ONNX model on first run; cached locally after that.
# No PyTorch required — ONNX Runtime only.
_model = TextEmbedding('sentence-transformers/all-MiniLM-L6-v2')

def embed_text(text: str) -> list[float]:
    return next(_model.embed([text])).tolist()


def clean_metadata(metadata: dict) -> dict:
    cleaned = {}
    for key, value in metadata.items():
        if value is None:
            continue
        if isinstance(value, np.generic):
            value = value.item()
        if isinstance(value, (list, tuple, set, dict)):
            value = json.dumps(value, sort_keys=True)
        cleaned[key] = value
    return cleaned


print('Model loaded: all-MiniLM-L6-v2 (fastembed / ONNX)')
print('Example vector length:', len(embed_text('Monza is fast and technical.')))
print('Clean metadata example:', clean_metadata({'driver': 'VER', 'sector': None, 'peak_dists': [10, 20]}))

/Users/ZaneQureshi/Desktop/ML/f1-analytics-/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded: all-MiniLM-L6-v2 (fastembed / ONNX)
Example vector length: 384
Clean metadata example: {'driver': 'VER', 'peak_dists': '[10, 20]'}


In [37]:
persist_dir = Path('../RAG_data_layers/chroma_store').resolve()
persist_dir.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(persist_dir))

collection_name = f"f1_qualifying_{SESSION_YEAR}_{SESSION_CIRCUIT.lower().replace(' ', '_')}"
existing_names = [collection.name for collection in client.list_collections()]
if collection_name in existing_names:
    client.delete_collection(collection_name)

collection = client.create_collection(
    name=collection_name,
    metadata={'hnsw:space': 'cosine'},
)

ids = []
documents = []
embeddings = []
metadatas = []

for chunk in chunks:
    ids.append(chunk.chunk_id)
    documents.append(chunk.text)
    embeddings.append(embed_text(chunk.text))
    metadatas.append(clean_metadata({'chunk_type': chunk.chunk_type, 'source': chunk.source, **chunk.metadata}))

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
)

print('Collection name:', collection_name)
print('Persist path:', persist_dir)
print('Documents written:', collection.count())
print('Embedding dimension used:', len(embeddings[0]) if embeddings else 0)
print('\nFirst stored metadata sample:')
print(metadatas[0])

Collection name: f1_qualifying_2024_monza
Persist path: /Users/ZaneQureshi/Desktop/ML/f1-analytics-/RAG_data_layers/chroma_store
Documents written: 857
Embedding dimension used: 384

First stored metadata sample:
{'chunk_type': 'circuit_overview', 'source': 'bacinger+tumftm', 'circuit': 'Autodromo Nazionale Monza', 'length_m': 5793, 'altitude_m': 142, 'first_gp_year': 1950, 'has_geometry': True}


In [38]:
def show_query_results(title: str, result: dict) -> None:
    rows = []
    documents = result.get('documents', [[]])[0]
    metadatas = result.get('metadatas', [[]])[0]
    distances = result.get('distances', [[]])[0]

    for rank, (doc, meta, distance) in enumerate(zip(documents, metadatas, distances), start=1):
        rows.append({
            'rank': rank,
            'distance': round(float(distance), 4),
            'chunk_type': meta.get('chunk_type'),
            'driver': meta.get('driver'),
            'event_id': meta.get('event_id'),
            'sector': meta.get('sector'),
            'text_preview': doc[:160].replace('\n', ' '),
        })

    print(title)
    display(pd.DataFrame(rows))


def query_with_local_embedding(collection, text: str, n_results: int = 5, where: dict | None = None):
    return collection.query(
        query_embeddings=[embed_text(text)],
        n_results=n_results,
        where=where,
    )


reloaded_client = chromadb.PersistentClient(path=str(persist_dir))
reloaded_collection = reloaded_client.get_collection(collection_name)
print('Reloaded collection count:', reloaded_collection.count())

# 1) Broad retrieval across the full collection
broad_result = query_with_local_embedding(
    reloaded_collection,
    'why did the lap get slower or get deleted',
    n_results=5,
)
show_query_results('Broad retrieval: race control / lap invalidation', broad_result)

# 2) Metadata-filtered retrieval for race control messages only
race_control_result = query_with_local_embedding(
    reloaded_collection,
    'why was the session red flagged',
    n_results=5,
    where={'chunk_type': 'race_control_event'},
)
show_query_results('Filtered retrieval: race control events only', race_control_result)

# 3) Driver-specific retrieval using a real driver value from the corpus
sample_driver = next((chunk.metadata.get('driver') for chunk in chunks if chunk.metadata.get('driver')), None)
if sample_driver:
    driver_result = query_with_local_embedding(
        reloaded_collection,
        f'how did {sample_driver} qualify and where was the time lost',
        n_results=5,
        where={
            '$and': [
                {'chunk_type': 'driver_lap_summary'},
                {'driver': sample_driver},
            ]
        },
    )
    show_query_results(f'Filtered retrieval: driver summary for {sample_driver}', driver_result)
else:
    print('No driver metadata value was available for the driver-specific retrieval test.')

Reloaded collection count: 857
Broad retrieval: race control / lap invalidation


,rank,distance,chunk_type,driver,event_id,sector,text_preview
0,1,0.564,session_pace_evolution,NOR,None,None,Pace Evolution: NOR Event: Monza 2024 Qualifying Total l...
1,2,0.617,session_pace_evolution,MAG,None,None,Pace Evolution: MAG Event: Monza 2024 Qualifying Total l...
2,3,0.624,session_pace_evolution,BOT,None,None,Pace Evolution: BOT Event: Monza 2024 Qualifying Total l...
3,4,0.624,session_pace_evolution,GAS,None,None,Pace Evolution: GAS Event: Monza 2024 Qualifying Total l...
4,5,0.624,session_pace_evolution,ZHO,None,None,Pace Evolution: ZHO Event: Monza 2024 Qualifying Total l...


Filtered retrieval: race control events only


,rank,distance,chunk_type,driver,event_id,sector,text_preview
0,1,0.564,race_control_event,None,None,None,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
1,2,0.564,race_control_event,None,None,None,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
2,3,0.566,race_control_event,None,None,None,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
3,4,0.582,race_control_event,None,None,None,Race Control Event — Monza 2024 Qualifying Time: 2024-08...
4,5,0.595,race_control_event,None,None,None,Race Control Event — Monza 2024 Qualifying Time: 2024-08...


Filtered retrieval: driver summary for NOR


,rank,distance,chunk_type,driver,event_id,sector,text_preview
0,1,0.738,driver_lap_summary,NOR,None,None,Qualifying Lap Summary: NOR (McLaren) Event: Monza 2024 ...


In [39]:
from rag_retrieval import build_question_prompt

question = f'Why did Piastri lose pole to Norris at {SESSION_CIRCUIT} {SESSION_YEAR}?'
question_packet = build_question_prompt(
    collection=reloaded_collection,
    question=question,
    embed_fn=embed_text,
    n_results=5,
    max_chunks_in_prompt=5,
)

print('Question:', question_packet['question'])
print('Retrieved chunks:', len(question_packet['retrieved_chunks']))

display(pd.DataFrame([
    {
        'rank': chunk['rank'],
        'distance': round(chunk['distance'], 4),
        'chunk_type': chunk['metadata'].get('chunk_type'),
        'driver': chunk['metadata'].get('driver'),
        'event_id': chunk['metadata'].get('event_id'),
        'text_preview': chunk['text'][:120].replace('\n', ' '),
    }
    for chunk in question_packet['retrieved_chunks']
]))

print('\n--- Prompt preview ---')
print(question_packet['prompt'][:3000])

Question: Why did Piastri lose pole to Norris at Monza 2024?
Retrieved chunks: 5


,rank,distance,chunk_type,driver,event_id,text_preview
0,1,0.512,head_to_head_event,None,S4,Head-to-Head at S4 (straight): PIA vs RUS Event: Monza 2...
1,2,0.513,head_to_head_event,None,S4,Head-to-Head at S4 (straight): PIA vs SAI Event: Monza 2...
2,3,0.525,head_to_head_event,None,T2,Head-to-Head at T2 (corner): PIA vs RUS Event: Monza 202...
3,4,0.530,head_to_head_event,None,S4,Head-to-Head at S4 (straight): PIA vs LEC Event: Monza 2...
4,5,0.531,head_to_head_event,None,T5,Head-to-Head at T5 (corner): PIA vs RUS Event: Monza 202...



--- Prompt preview ---
You are an F1 qualifying analyst. Answer the question using only the retrieved context. If the context does not fully support the answer, say what is missing.

Question: Why did Piastri lose pole to Norris at Monza 2024?

Retrieved context:
[1] head_to_head_event | S4 | fastf1
Head-to-Head at S4 (straight): PIA vs RUS
Event: Monza 2024 Qualifying
PIA P2 (McLaren) | RUS P3 (Mercedes)
Distance window: 2148m – 2592m

  Entry speed: PIA 129.4km/h | RUS 131.0km/h | advantage: RUS +1.6km/h
  Top speed: PIA 262.0km/h | RUS 262.0km/h | advantage: equal
DRS: PIA closed | RUS closed
Braking point into next corner:
  PIA: 2453m at 240.0 km/h
  RUS: 2423m at 261.0 km/h
  Exit speed: PIA 229.0km/h | RUS 227.0km/h | advantage: PIA +2.0km/h

[2] head_to_head_event | S4 | fastf1
Head-to-Head at S4 (straight): PIA vs SAI
Event: Monza 2024 Qualifying
PIA P2 (McLaren) | SAI P5 (Ferrari)
Distance window: 2148m – 2592m

  Entry speed: PIA 129.4km/h | SAI 126.7km/h | advantage: PIA +

---
## Summary

You've now successfully:
- Pulled circuit metadata and geometry from two GitHub sources
- Loaded a qualifying session via FastF1 (lap times, telemetry, sector splits)
- Pulled weather and race control context from OpenF1
- Combined all three into structured context blocks per driver
- Turned those chunks into ChromaDB embeddings with metadata preserved for filtering
- Verified the store with broad retrieval and metadata-filtered smoke tests

This notebook now covers the full data-to-vector-store path for the qualifying RAG layer.